#### Shared Variables 

In [1]:
import os

# Set the correct Java Home
os.environ['JAVA_HOME'] = "/opt/homebrew/opt/openjdk@17/libexec/openjdk.jdk/Contents/Home"

# Set the Spark Home
os.environ['SPARK_HOME'] = "/opt/homebrew/opt/apache-spark/libexec"

In [2]:
from pyspark.sql import SparkSession

In [4]:
spark = SparkSession.builder.appName("Demo-Accumulators").getOrCreate()

print("Spark Version :: ",spark.version)

Spark Version ::  4.0.0


In [5]:
sc = spark.sparkContext

In [12]:
data = [1,2,3,4,5,6,7,8]

rdd = sc.parallelize(data)

rdd.getNumPartitions()
rdd2 = rdd.repartition(10)

rdd3 = rdd.coalesce(5)
#rdd2.getNumPartitions()
rdd3.getNumPartitions()

5

In [13]:
# create an accumulator

even_numbers = sc.accumulator(0)

data = sc.parallelize(range(1,10))

def count_even(x):
    if x % 2 == 0:
        even_numbers.add(1)

#map 
# foreach 

data.foreach(count_even)

print("Total even numbers : ",even_numbers.value )






Total even numbers :  4


In [14]:
# Corrupt records 

corrupt_records = sc.accumulator(0)

data = sc.parallelize([ "a,1",
                        "b,2",
                        "c,",
                        "d,4",
                        "e,"])

def validate_record(record):
    parts = record.split(",")
    if len(parts) != 2 or not parts[1] or not parts[0]:
        corrupt_records.add(1)


data.foreach(validate_record)

print(f"Number of corrupt records : {corrupt_records.value}")

     


Number of corrupt records : 2


In [15]:
data = [100,250,75,500,300]

total_size = sc.accumulator(0)

file_sizes = sc.parallelize(data) # rdd 

file_sizes.foreach(lambda x: total_size.add(x))

print(f"Total size of files : {total_size.value}")



Total size of files : 1225


#### Playing with partitions 

In [16]:
data = sc.parallelize(range(1,11),4)

data.getNumPartitions()

4

In [17]:
# To get the contents of each partition 
# glom()

partition_data = data.glom().collect()

print(type(partition_data)) # list 

print(partition_data) # list of list 




<class 'list'>
[[1, 2], [3, 4, 5], [6, 7], [8, 9, 10]]


In [ ]:
# enumerate 

for i , partition in enumerate(partition_data):
    print(f"Partition {i+1} : {partition}")


#same thing 
for i  in range(len(partition_data)):
  
    print(f"Partition {i+1} : {partition_data[i]}")

Partition 1 : [1, 2]
Partition 2 : [3, 4, 5]
Partition 3 : [6, 7]
Partition 4 : [8, 9, 10]
Partition 1 : [1, 2]
Partition 2 : [3, 4, 5]
Partition 3 : [6, 7]
Partition 4 : [8, 9, 10]


In [24]:
# mapPartitions 

data = sc.parallelize(range(1,10),3)

def process_partition(iterator):
    count = 0 
    for x in iterator:
        count += 1
    yield count 

partitions_count = data.mapPartitions(process_partition).collect()       

print(f"Counts per Partitions :: {partitions_count}")


Counts per Partitions :: [3, 3, 3]


In [25]:
# foreachPartition() Action 

data = sc.parallelize(range(1,7),2)

def write_partition(iterator):
    with open("september_one.txt","a") as f:
        for item in iterator:
            f.write(f"{item}\n")

data.foreachPartition(write_partition)             


#### BroadCast Variables 

In [27]:
transactions = sc.parallelize([(1,'A',100),
                              (2,'B',200),
                              (3,'C',150),
                              (4,'D',50)])

popular_products = ['A','C']
 

broadcast_products = sc.broadcast(popular_products)


filtered_tranactions = transactions.filter(lambda x : x[1] in  broadcast_products.value)



print(f"Filtered transactions : {filtered_tranactions.collect()}")





Filtered transactions : [(1, 'A', 100), (3, 'C', 150)]
